In [ ]:
import sys
from pathlib import Path
from typing import cast

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr


# ============================================================
# 0. Daten, Labels und Gewichte laden
# ============================================================

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.saving_utils import DatasetBundle


data_discov_full = DatasetBundle.load(
    project_root / "data/preprocessed/data_discov_full"
)

data_valid_full = DatasetBundle.load(
    project_root / "data/preprocessed/data_valid_full"
)

data_discov_full.meta = cast(
    pd.DataFrame,
    data_discov_full.meta
)

data_valid_full.meta = cast(
    pd.DataFrame,
    data_valid_full.meta
)


# Clusterlabels laden
labels_discov = pd.read_parquet(
    project_root
    / "outputs/gmm_clustering/cluster_pred_discov.parquet"
)["label"].reset_index(drop=True)

labels_valid = pd.read_parquet(
    project_root
    / "outputs/gmm_clustering/cluster_pred_valid.parquet"
)["label"].reset_index(drop=True)


# Surveygewichte laden
sample_weight_discov = pd.to_numeric(
    data_discov_full.data["wichtn"],
    errors="coerce"
).to_numpy(dtype=float)

sample_weight_valid = pd.to_numeric(
    data_valid_full.data["wichtn"],
    errors="coerce"
).to_numpy(dtype=float)


# Prüfen, ob Daten, Labels und Gewichte zusammenpassen
if not (
    len(data_discov_full.data)
    == len(labels_discov)
    == len(sample_weight_discov)
):
    raise ValueError(
        "Discovery-Daten, Labels und Gewichte haben "
        "unterschiedliche Längen."
    )

if not (
    len(data_valid_full.data)
    == len(labels_valid)
    == len(sample_weight_valid)
):
    raise ValueError(
        "Validation-Daten, Labels und Gewichte haben "
        "unterschiedliche Längen."
    )


# ============================================================
# 1. Kontinuierliche und ordinale Variablen auswählen
# ============================================================

profile_variables = (
    data_discov_full.meta
    .loc[
        lambda df: df["type"].isin(
            ["continuous", "ordinal"]
        ),
        "name"
    ]
    .tolist()
)

# Nur Variablen verwenden, die in beiden Wellen vorhanden sind.
# Die Surveygewichtung selbst wird nicht als Profilvariable verwendet.
profile_variables = [
    variable
    for variable in profile_variables
    if variable != "wichtn"
    and variable in data_discov_full.data.columns
    and variable in data_valid_full.data.columns
]

print(
    f"{len(profile_variables)} gemeinsame kontinuierliche "
    f"und ordinale Variablen gefunden."
)


# ============================================================
# 2. Daten numerisch konvertieren
# ============================================================

def to_numeric_frame(
    data: pd.DataFrame,
    variables: list[str]
) -> pd.DataFrame:
    """
    Konvertiert die ausgewählten Variablen in numerische Werte.
    Nicht konvertierbare Werte werden als NaN gespeichert.
    """
    result = {}

    for variable in variables:
        series = data[variable]

        if isinstance(
            series.dtype,
            pd.CategoricalDtype
        ):
            series = series.astype(str)

        result[variable] = pd.to_numeric(
            series,
            errors="coerce"
        )

    return pd.DataFrame(
        result,
        index=data.index
    )


disc_data = to_numeric_frame(
    data_discov_full.data,
    profile_variables
).reset_index(drop=True)

valid_data = to_numeric_frame(
    data_valid_full.data,
    profile_variables
).reset_index(drop=True)


# ============================================================
# 3. Gewichtete Clustermittelwerte berechnen
# ============================================================

def weighted_mean(
    values: pd.Series | np.ndarray,
    weights: np.ndarray
) -> float:
    """
    Berechnet einen gewichteten Mittelwert unter Ausschluss
    fehlender Werte und ungültiger Gewichte.
    """
    values = np.asarray(
        values,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    usable = (
        np.isfinite(values)
        & np.isfinite(weights)
        & (weights > 0)
    )

    if usable.sum() == 0:
        return np.nan

    return float(
        np.average(
            values[usable],
            weights=weights[usable]
        )
    )


def weighted_cluster_means(
    data: pd.DataFrame,
    labels: pd.Series,
    weights: np.ndarray
) -> pd.DataFrame:
    """
    Berechnet für jedes Cluster die gewichteten Mittelwerte
    aller Profilvariablen.
    """
    labels = pd.Series(
        labels
    ).reset_index(drop=True)

    data = data.reset_index(drop=True)

    weights = np.asarray(
        weights,
        dtype=float
    )

    if not (
        len(data)
        == len(labels)
        == len(weights)
    ):
        raise ValueError(
            "Daten, Labels und Gewichte haben "
            "unterschiedliche Längen."
        )

    results = {}

    for cluster in sorted(
        labels.dropna().unique()
    ):
        mask = labels == cluster

        results[cluster] = {
            variable: weighted_mean(
                data.loc[mask, variable],
                weights[mask.to_numpy()]
            )
            for variable in data.columns
        }

    result = pd.DataFrame.from_dict(
        results,
        orient="index"
    )

    result.index.name = "cluster"

    return result


disc_cluster_means = weighted_cluster_means(
    disc_data,
    labels_discov,
    sample_weight_discov
)

valid_cluster_means = weighted_cluster_means(
    valid_data,
    labels_valid,
    sample_weight_valid
)


# ============================================================
# 4. Discovery-Mittelwerte und Discovery-SDs berechnen
# ============================================================

def weighted_mean_sd(
    data: pd.DataFrame,
    weights: np.ndarray
) -> tuple[pd.Series, pd.Series]:
    """
    Berechnet für jede Variable den gewichteten Mittelwert und
    die gewichtete Populationsstandardabweichung.
    """
    means = {}
    standard_deviations = {}

    weights = np.asarray(
        weights,
        dtype=float
    )

    for variable in data.columns:
        values = data[
            variable
        ].to_numpy(dtype=float)

        usable = (
            np.isfinite(values)
            & np.isfinite(weights)
            & (weights > 0)
        )

        x = values[usable]
        w = weights[usable]

        if len(x) == 0 or w.sum() <= 0:
            means[variable] = np.nan
            standard_deviations[variable] = np.nan
            continue

        mean = np.average(
            x,
            weights=w
        )

        variance = np.average(
            (x - mean) ** 2,
            weights=w
        )

        means[variable] = mean
        standard_deviations[variable] = np.sqrt(
            variance
        )

    return (
        pd.Series(
            means,
            dtype=float
        ),
        pd.Series(
            standard_deviations,
            dtype=float
        )
    )


discovery_mean, discovery_sd = weighted_mean_sd(
    disc_data,
    sample_weight_discov
)


# Nur Variablen mit gültiger positiver Discovery-SD verwenden
usable_variables = [
    variable
    for variable in profile_variables
    if pd.notna(discovery_mean[variable])
    and pd.notna(discovery_sd[variable])
    and np.isfinite(discovery_sd[variable])
    and discovery_sd[variable] > 0
]

excluded_variables = [
    variable
    for variable in profile_variables
    if variable not in usable_variables
]

if len(usable_variables) < 3:
    raise ValueError(
        "Für den Profilvergleich sind weniger als drei "
        "verwendbare Variablen vorhanden."
    )

print(
    f"{len(usable_variables)} Variablen werden für den "
    f"Profilvergleich verwendet."
)

print("\nAusgeschlossene Variablen:")
print(excluded_variables)


# ============================================================
# 5. Clustergrößen bestimmen
# ============================================================

cluster_sizes_discov = (
    labels_discov
    .value_counts()
    .sort_index()
    .rename("n")
)

cluster_sizes_valid = (
    labels_valid
    .value_counts()
    .sort_index()
    .rename("n")
)

print("\nClustergrößen Discovery:")
display(
    cluster_sizes_discov.to_frame()
)

print("Clustergrößen Validation:")
display(
    cluster_sizes_valid.to_frame()
)


# ============================================================
# 6. Profilvergleich und Heatmap
# ============================================================

def compare_cluster_profiles(
    disc_means: pd.DataFrame,
    valid_means: pd.DataFrame,
    title: str
) -> dict:
    """
    Standardisiert die Clusterprofile mit der Discovery-Skala
    und berechnet für jedes Discovery- und Validation-Cluster
    die Spearman-Korrelation der standardisierten Profile.

    Die vollständige Korrelationsmatrix wird anschließend
    ohne Matching oder Umordnung als Heatmap dargestellt.
    """

    if disc_means.empty:
        raise ValueError(
            f"{title}: Keine Discovery-Cluster vorhanden."
        )

    if valid_means.empty:
        raise ValueError(
            f"{title}: Keine Validation-Cluster vorhanden."
        )


    # --------------------------------------------------------
    # Beide Wellen mit derselben Discovery-Skala standardisieren
    # --------------------------------------------------------

    disc_profiles_z = (
        disc_means[usable_variables]
        - discovery_mean[usable_variables]
    ) / discovery_sd[usable_variables]

    valid_profiles_z = (
        valid_means[usable_variables]
        - discovery_mean[usable_variables]
    ) / discovery_sd[usable_variables]


    # --------------------------------------------------------
    # Spearman-Korrelationen
    # --------------------------------------------------------

    spearman_matrix = pd.DataFrame(
        index=disc_profiles_z.index,
        columns=valid_profiles_z.index,
        dtype=float
    )

    for discovery_cluster in disc_profiles_z.index:

        for validation_cluster in valid_profiles_z.index:

            x = disc_profiles_z.loc[
                discovery_cluster
            ]

            y = valid_profiles_z.loc[
                validation_cluster
            ]

            complete = (
                x.notna()
                & y.notna()
            )

            x_complete = x.loc[
                complete
            ]

            y_complete = y.loc[
                complete
            ]

            if (
                len(x_complete) < 3
                or x_complete.nunique() < 2
                or y_complete.nunique() < 2
            ):
                correlation = np.nan

            else:
                correlation = spearmanr(
                    x_complete,
                    y_complete
                ).statistic

            spearman_matrix.loc[
                discovery_cluster,
                validation_cluster
            ] = correlation


    # --------------------------------------------------------
    # Korrelationsmatrix ausgeben
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    print("\nSpearman-Korrelationen:")

    display(
        spearman_matrix.round(3)
    )


    # --------------------------------------------------------
    # Heatmap der vollständigen Korrelationsmatrix
    # --------------------------------------------------------

    figure_width = max(
        8,
        len(
            spearman_matrix.columns
        ) * 0.45
    )

    figure_height = max(
        6,
        len(
            spearman_matrix.index
        ) * 0.40
    )

    plt.figure(
        figsize=(
            figure_width,
            figure_height
        )
    )

    image = plt.imshow(
        spearman_matrix.to_numpy(
            dtype=float
        ),
        aspect="auto",
        vmin=-1,
        vmax=1,
        cmap="coolwarm"
    )

    plt.colorbar(
        image,
        label="Spearman correlation"
    )

    plt.xticks(
        ticks=np.arange(
            len(
                spearman_matrix.columns
            )
        ),
        labels=[
            f"V {cluster}"
            for cluster
            in spearman_matrix.columns
        ],
        rotation=90
    )

    plt.yticks(
        ticks=np.arange(
            len(
                spearman_matrix.index
            )
        ),
        labels=[
            f"D {cluster}"
            for cluster
            in spearman_matrix.index
        ]
    )

    plt.xlabel(
        "Validation clusters"
    )

    plt.ylabel(
        "Discovery clusters"
    )

    plt.title(
        f"{title}\n"
        "Spearman correlations between cluster profiles"
    )

    plt.tight_layout()
    plt.show()


    return {
        "disc_profiles_z": disc_profiles_z,
        "valid_profiles_z": valid_profiles_z,
        "spearman_matrix": spearman_matrix
    }


# ============================================================
# 7. Vergleich mit allen Clustern
# ============================================================

results_all_clusters = compare_cluster_profiles(
    disc_means=disc_cluster_means,
    valid_means=valid_cluster_means,
    title=(
        "Similarity of standardized cluster profiles: "
        "all clusters"
    )
)


# ============================================================
# 8. Sensitivitätsanalyse:
#    nur Cluster mit mindestens 100 Personen
# ============================================================

min_cluster_size = 100

large_disc_clusters = (
    cluster_sizes_discov[
        cluster_sizes_discov
        >= min_cluster_size
    ]
    .index
    .tolist()
)

large_valid_clusters = (
    cluster_sizes_valid[
        cluster_sizes_valid
        >= min_cluster_size
    ]
    .index
    .tolist()
)

print(
    f"\nDiscovery-Cluster mit mindestens "
    f"{min_cluster_size} Personen:"
)

print(
    large_disc_clusters
)

print(
    f"Validation-Cluster mit mindestens "
    f"{min_cluster_size} Personen:"
)

print(
    large_valid_clusters
)


disc_cluster_means_100 = (
    disc_cluster_means
    .loc[
        large_disc_clusters
    ]
    .copy()
)

valid_cluster_means_100 = (
    valid_cluster_means
    .loc[
        large_valid_clusters
    ]
    .copy()
)


results_clusters_100 = compare_cluster_profiles(
    disc_means=disc_cluster_means_100,
    valid_means=valid_cluster_means_100,
    title=(
        "Similarity of standardized cluster profiles: "
        "clusters with at least 100 observations"
    )
)